# Breast Cancer: EDA and Logistic Regression (Ridge and Lasso)

## 1. Load data

In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore")

# built-in dataset, no file needed. target: 1 = benign, 0 = malignant
raw = load_breast_cancer(as_frame=True)
data = raw.frame

print(data.shape)
print(data.head())

## 2. Missing values

In [ ]:
print("Missing values:", data.isnull().sum().sum())
print("Duplicate rows:", data.duplicated().sum())

No missing values here, but the imputer is still kept in the pipeline so it works on messy data too.

## 3. EDA

In [ ]:
print(data['target'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(x='target', data=data, ax=axes[0])
axes[0].set_title('Class balance (0 = malignant, 1 = benign)')
sns.histplot(data=data, x='mean radius', hue='target', bins=30, ax=axes[1])
axes[1].set_title('Mean radius by class')
plt.show()

In [ ]:
cols = ['mean radius', 'mean texture', 'mean area', 'mean concavity', 'mean smoothness', 'mean symmetry']

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, cols):
    sns.boxplot(x='target', y=col, data=data, ax=ax)
    ax.set_title(col + ' by class')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(data.iloc[:, :10].join(data['target']).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix (mean features + target)')
plt.show()

## 4. Target and split

The target is already binary (1 = benign, 0 = malignant), so no conversion is needed. 80% is used for training and 20% for validation.

In [ ]:
x = data.drop(columns='target')
y = data['target']

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(x_train.shape, x_val.shape)

## 5. Pipeline

In [ ]:
# all features are numeric, so one simple pipeline is enough
preprocess = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

## 6. Ridge (L2)

In [ ]:
param_grid = {'model__C': [0.01, 0.1, 1, 10]}

ridge_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(penalty='l2', solver='liblinear'))
])

ridge_grid = GridSearchCV(ridge_pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
ridge_grid.fit(x_train, y_train)

print("Best C:", ridge_grid.best_params_['model__C'])
print("Best CV accuracy:", ridge_grid.best_score_)

## 7. Lasso (L1)

In [ ]:
lasso_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(penalty='l1', solver='liblinear'))
])

lasso_grid = GridSearchCV(lasso_pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
lasso_grid.fit(x_train, y_train)

print("Best C:", lasso_grid.best_params_['model__C'])
print("Best CV accuracy:", lasso_grid.best_score_)

## 8. Compare

In [ ]:
models = {'Ridge': ridge_grid, 'Lasso': lasso_grid}

for name, grid in models.items():
    pred = grid.predict(x_val)
    print(name)
    print("  Accuracy:", round(accuracy_score(y_val, pred), 4))
    print("  F1:      ", round(f1_score(y_val, pred), 4))
    print("  ROC AUC: ", round(roc_auc_score(y_val, grid.predict_proba(x_val)[:, 1]), 4))
    print("  Confusion matrix:")
    print(confusion_matrix(y_val, pred))

## 9. Feature importance

In [ ]:
# Lasso sets some coefficients to exactly 0, Ridge only shrinks them
coefs = pd.DataFrame({
    'Ridge': ridge_grid.best_estimator_['model'].coef_[0],
    'Lasso': lasso_grid.best_estimator_['model'].coef_[0]
}, index=x.columns)

print("Features dropped by Lasso:", (coefs['Lasso'] == 0).sum(), "of", len(coefs))

top = coefs.reindex(coefs['Lasso'].abs().sort_values(ascending=False).index).head(10)
top.plot(kind='barh', figsize=(8, 5), title='Top 10 coefficients')
plt.gca().invert_yaxis()
plt.show()

## 10. Predict on new data

In [ ]:
best_model = ridge_grid if ridge_grid.score(x_val, y_val) >= lasso_grid.score(x_val, y_val) else lasso_grid
print("Best model:", "Ridge" if best_model is ridge_grid else "Lasso")

# example: take 5 rows as if they were new patients
new_data = x_val.head(5)
new_pred = best_model.predict(new_data)

result = pd.DataFrame({'Predicted': new_pred, 'Actual': y_val.head(5).values}, index=new_data.index)
result['Predicted'] = result['Predicted'].map({0: 'malignant', 1: 'benign'})
result['Actual'] = result['Actual'].map({0: 'malignant', 1: 'benign'})
print(result)